### Local File Agent

로컬 파일을 검색하고 읽고 쓰는 Tool을 정의한 뒤 Agent가 여러 Tool을 선택해 작업하는 과정을 확인한다. 파일 처리 로직은 제공된 함수를 사용하고, Tool의 이름과 설명은 직접 작성한다.

In [5]:
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

MODEL_NAME = "gemini-3.6-flash"

### 실습 환경

현재 작업 폴더에 예제 파일을 준비한다. 아래 함수가 실제 파일 검색, 읽기, 쓰기를 처리한다. 모든 작업은 현재 작업 폴더의 `.txt` 파일로 제한된다.

In [2]:
WORKSPACE = Path.cwd().resolve()

sample_files = {
    "langgraph.txt": "LangGraph는 상태 기반 워크플로를 그래프로 구성한다.",
    "react.txt": "ReAct Agent는 추론과 도구 사용을 반복해 문제를 해결한다.",
    "tools.txt": "Tool의 이름과 설명은 모델의 도구 선택에 영향을 준다.",
}

for filename, content in sample_files.items():
    path = WORKSPACE / filename
    if not path.exists():
        path.write_text(content, encoding="utf-8")

print(f"실습 폴더: {WORKSPACE}")
print(f"파일 목록: {[path.name for path in WORKSPACE.glob('*.txt')]}")

실습 폴더: C:\Users\snail\OneDrive\바탕 화면\aim-ai-agent-my\langchain_\12-react-agent-practice
파일 목록: ['langgraph.txt', 'react.txt', 'tools.txt']


In [3]:
# 파일 경로가 작업 폴더 안의 txt 파일인지 확인한다.
def _resolve_path(filename: str) -> Path:
    # 작업 폴더와 파일 이름을 연결하고 절대 경로로 변환한다.
    path = (WORKSPACE / filename).resolve()
    # 상위 폴더로 벗어나는 경로의 사용을 막는다.
    if path.parent != WORKSPACE:
        raise ValueError("현재 작업 폴더 안의 파일만 사용할 수 있습니다.")
    # suffix는 파일의 확장자를 반환한다.
    if path.suffix.lower() != ".txt":
        raise ValueError("txt 파일만 사용할 수 있습니다.")
    return path


# 파일 이름과 내용에서 검색어가 포함된 txt 파일을 찾는다.
def search_files_impl(keyword: str) -> str:
    # 앞뒤 공백을 제거하고 대소문자 구분 없이 검색할 수 있게 변환한다.
    keyword = keyword.strip().lower()
    matched = []

    # glob("*.txt")로 작업 폴더의 모든 txt 파일을 순회한다.
    for path in WORKSPACE.glob("*.txt"):
        # 파일 내용을 UTF-8 문자열로 읽는다.
        content = path.read_text(encoding="utf-8")
        # 검색어가 파일 이름이나 내용에 포함되면 결과에 추가한다.
        if keyword in path.name.lower() or keyword in content.lower():
            matched.append(path.name)

    # 파일 이름을 정렬해 줄바꿈으로 연결하고, 결과가 없으면 안내 문구를 반환한다.
    return "\n".join(sorted(matched)) or "검색 결과가 없습니다."


# 파일의 존재 여부와 크기를 확인한 뒤 내용을 읽는다.
def read_file_impl(filename: str) -> str:
    # 안전한 파일 경로인지 먼저 검사한다.
    path = _resolve_path(filename)
    # exists()로 파일이 실제로 존재하는지 확인한다.
    if not path.exists():
        return "파일을 찾을 수 없습니다."
    # stat().st_size로 파일 크기를 바이트 단위로 확인한다.
    if path.stat().st_size > 10_000:
        return "파일이 너무 커서 읽을 수 없습니다."
    # 검사를 통과한 파일의 전체 내용을 문자열로 반환한다.
    return path.read_text(encoding="utf-8")


# 내용의 길이를 확인한 뒤 txt 파일로 저장한다.
def write_file_impl(filename: str, content: str) -> str:
    # 안전한 파일 경로인지 먼저 검사한다.
    path = _resolve_path(filename)
    # 지나치게 긴 내용이 저장되지 않도록 글자 수를 제한한다.
    if len(content) > 5_000:
        return "내용이 너무 길어서 저장할 수 없습니다."
    # write_text()로 문자열을 UTF-8 형식의 파일에 저장한다.
    path.write_text(content, encoding="utf-8")
    return f"{path.name} 파일을 작성했습니다."

In [4]:
print("[파일 검색]")
print(search_files_impl("ReAct"))

print("\n[파일 읽기]")
print(read_file_impl("react.txt"))

print("\n[파일 쓰기]")
print(write_file_impl("function_test.txt", "함수 실행 결과를 확인합니다."))

print("\n[작성 결과 확인]")
print(read_file_impl("function_test.txt"))

[파일 검색]
react.txt

[파일 읽기]
ReAct Agent는 추론과 도구 사용을 반복해 문제를 해결한다.

[파일 쓰기]
function_test.txt 파일을 작성했습니다.

[작성 결과 확인]
함수 실행 결과를 확인합니다.


### Tool 정의 실습

제공된 `search_files_impl`, `read_file_impl`, `write_file_impl`을 호출하는 Tool 세 개를 정의한다.


In [ ]:
@tool(parse_docstring=True)
def search_files_impl_tool(keyword: str) -> str:
    """파일 이름과 내용에서 검색어가 포함된 txt 파일을 찾는다."""
    return search_files_impl(keyword)

@tool(parse_docstring=True)
def read_file_impl_tool(filename: str) -> str:
    """파일의 존재 여부와 크기를 확인한 뒤 내용을 읽는다."""
    return read_file_impl(filename)

@tool(parse_docstring=True)
def write_file_impl_tool(filename: str, content: str) -> str:
    """내용의 길이를 확인한 뒤 txt 파일로 저장한다."""
    return write_file_impl(filename, content)

tools = [
    search_files_impl_tool,
    read_file_impl_tool,
    write_file_impl_tool
]


In [ ]:
# 강사님 풀이
@tool(parse_docstring=True)
def search_files_tool(keyword: str):
    """파일 이름이나 내용에 검색어가 포함된 file을 찾는 함수.
    
    Args:
        keyword: 검색어"""

    return search_files_impl(keyword)

@tool(parse_docstring=True)
def read_files_tool(filename: str):
    """파일 이름을 바탕으로 해당 파일의 내용일 읽는 함수..
    
    Args:
        filename: 파일 이름"""

    return read_file_impl(filename)


@tool(parse_docstring=True)
def write_files_tool(filename: str, content: str):
    """입력받은 content를 파일에 작성하는 함수.
    
    Args:
        filename: 파일 이름
        content : 내용
    """

    return write_file_impl(filename, content)

### Agent 구성

작성한 Tool을 사용하는 Agent를 생성한다.

In [11]:
llm = ChatGoogleGenerativeAI(model= MODEL_NAME)

agent = create_agent(
    model=llm,
    tools=tools
)

In [ ]:
# 강사님 풀이
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
tools = [search_files_tool, read_files_tool, write_files_tool]
agent = create_agent(llm, tools)

### Agent 실행

Agent가 검색, 읽기, 쓰기 Tool을 순서대로 선택하는지 확인한다.

In [12]:
question = "ReAct 관련 파일을 찾아 읽고, 핵심 내용을 summary.txt에 저장해 줘."

for state in agent.stream(
    {"messages": [{"role": "human", "content": question}]},
    stream_mode="values",
):
    message = state["messages"][-1]
    print(f"[{message.type}] {message.text}")
    if getattr(message, "tool_calls", None):
        print(f"  -> Tool 호출: {[call['name'] for call in message.tool_calls]}")
    print()

[human] ReAct 관련 파일을 찾아 읽고, 핵심 내용을 summary.txt에 저장해 줘.

[ai] 
  -> Tool 호출: ['search_files_impl_tool']

[tool] react.txt

[ai] 
  -> Tool 호출: ['read_file_impl_tool']

[tool] ReAct Agent는 추론과 도구 사용을 반복해 문제를 해결한다.

[ai] 
  -> Tool 호출: ['write_file_impl_tool']

[tool] summary.txt 파일을 작성했습니다.

[ai] ReAct 관련 파일인 `react.txt`를 찾아 확인한 후, 핵심 내용을 정리하여 `summary.txt`에 저장했습니다.

### 저장된 내용 (`summary.txt`)
> ReAct Agent는 추론과 도구 사용을 반복하여 문제를 해결한다.



In [14]:
summary_path = WORKSPACE / "summary.txt"

if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("summary.txt가 작성되지 않았습니다.")

ReAct Agent는 추론과 도구 사용을 반복하여 문제를 해결한다.
